# Chunked-Prefill



[SARATHI: Efficient LLM Inference by Piggybacking Decodes with Chunked Prefills](https://arxiv.org/pdf/2308.16369)

摘选 Abstration

1. Decoding 计算量不充分，memory-bound


> Large Language Model (LLM) inference consists of two distinct phases – prefill phase which processes the input prompt and decode phase which generates output tokens autoregressively. While the prefill phase effectively saturates GPU compute at small batch sizes, the decode phase results in low compute utilization as it generates one token at a time per request. The varying prefill and decode times also lead to imbalance across micro-batches when using pipeline parallelism, resulting in further inefficiency due to bubbles.


2. Decoding 投影搭 Prefill 便车

> We present SARATHI to address these challenges. SARATHI employs chunked-prefills, which splits a **prefill request into equal sized chunks, and decode-maximal batching**, which constructs a batch using a single prefill chunk and populates the **remaining slots with decodes**. During inference, the prefill chunk saturates GPU compute, while the decode requests ‘piggyback’ and cost up to an order of magnitude less compared to a decode-only batch. Chunked-prefills allows constructing multiple decode-maximal batches from a single prefill request, maximizing coverage of decodes that can piggyback. Furthermore, the uniform compute design of these batches ameliorates the imbalance between micro-batches, significantly reducing pipeline bubbles.

这一段还描述了一个特殊的 Feature

"prefill request into equal sized chunks, and decode-maximal batching", 

1. 所有 decoding 请求压成一个 batching,
2. 单一 Prefill 请求可以进行切片，切片数据称为一个 batch，切片是在 input-ids 序列规模上的。


## Part1: Proj 投影搭便车

解码过程特性：

1. Prefill: Compute-bound
2. Decoding: Memory-bound，我们熟知的计算 attention 前要拉取大量的 KVCache，存在显著的 memory 访问的开销。

另外从投影分析：

在 Decoding 时，输入为 next_token, 此时需要拉取一个Attn里的 Wq、Wk、Wv 做投影，或者 FFN 里的 W 权重，计算形式为

`x(1xd) @ Wq(dxd)`

可见我们频繁拉取大的权重矩阵到 SRAM 中，只是做简单的运算，这种访存开销是不经济的。而 Prefill 则是产生了充分的计算的，如：

```
X(Lxd) @ Wq(dxd)
```

此时如果我们定义两个请求：其输入为 
```
req1 (prefill stage): X_req1 [1000xd]
req2 (decoding stage): x_req2 [1xd]
```

我们进行拼接为
```
X_cp = torch.cat( [X_req1, x_req2], dim=0)
Q_cp = X_cp @ Wq
Q_req1, q_req2 <- split(Q_cp)
```

这种处理技巧我们称为 Chunked—Prefill。 为什么不叫 Chunked-Decoding?

1. Prefill 在投影计算中是矩阵乘高效的
2. Decoding 搭了 Prefill 的便车
3. Chunked 的定义，我理解是 `X_cp` 对应有多块（chunked）数据来源

可以理解为通过融合 PD 阶段共性计算，减少 memory-visited 开销，从而提速。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple, Optional, Any

torch.manual_seed(42)

In [2]:
d = 2048

# Prefill 和 Decoding 各一个请求
L_p = 1024
L_d = 1 

X_p = torch.randn(L_p, d)
X_d = torch.randn(L_d, d)

W = torch.randn(d,d)

# combine
X_pd = torch.cat( (X_p, X_d), dim = 0)

Y_pd = X_pd @ W

Yp = Y_pd[:L_p, :]
Yd = Y_pd[L_p:, :]

In [3]:
Wq = nn.Linear(d, d)
    
def ChunkPrefillLinearForward(W, XP, XD):
    with torch.no_grad():
        L_p, d = XP.shape
        XPD = torch.cat((XP, XD), dim = 0)
        YPD = W(XPD)
        return YPD[:L_p, :], YPD[L_p:, :]

Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([1024, 2048])
torch.Size([1, 2048])


In [4]:
### 通用实现

# config 
d = 2048
bsz_p = 3 # prefill请求数量
bsz_d = 20 # decoding请求数量
seq_p = 1024
seq_d = 1 

# data
Wq = nn.Linear(d, d)

def ChunkPrefillLinearForward(W, XP, XD):
    """
    更通用的 Chunk Prefill, 可处理 PD batch size 不同的情况
    """
    
    BP, LP, D = XP.shape
    BD, LD, D = XD.shape
    
    with torch.no_grad():
        XP = XP.reshape(BP*LP, D)
        XD = XD.reshape(BD*LD, D)
        XPD = torch.cat((XP, XD), dim = 0)
        YPD = W(XPD)

        YP = YPD[:BP*LP].reshape(BP, LP, D)
        YD = YPD[BP*LP:].reshape(BD, LD, D)

        return YP, YD

X_p = torch.randn(bsz_p, seq_p, d) 
X_d = torch.randn(bsz_d, seq_d, d)

Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([3, 1024, 2048])
torch.Size([20, 1, 2048])


## Part2. Chunked-Prefill


当一个请求长度为 20, page 长度为 8, 那么可以切成 3 个 page。 

1. 前向计算逻辑？
2. 此计算与 context-parallelsim 的区别是什么？


### Basic Prefill 实现

In [5]:
bsz = 1
seq_len = 20 
page_size = 8
dim = 16
vocab_size = 100

In [6]:
### Basic Model

def attention_kernel(Q, K, V):
    S = Q @ K.transpose(1,2)
    P = F.softmax(S, dim = -1)
    Z = P @ V
    return Z

class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.Wq = nn.Linear(dim, dim)
        self.Wk = nn.Linear(dim, dim)
        self.Wv = nn.Linear(dim, dim)
        self.Wo = nn.Linear(dim, dim)
        
    def forward(self, X, KVCache=None):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        Z = attention_kernel(Q,K,V)
        O = self.Wo(Z)
        return O, [K,V]

class XDGModel(nn.Module):
    def __init__(self, dim, vocab_size, attention_layer):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, dim)
        self.decoder = attention_layer(dim)
        self.lm_head = nn.Linear(dim, vocab_size)
    def forward(self, x, KVCache=None):
        X = self.embd(x)
        H, KV = self.decoder(X, KVCache=KVCache)
        logits = self.lm_head(H)
        return logits, KV

In [7]:
model = XDGModel(dim, vocab_size, attention_layer=Attention)
# X = torch.randn(bsz, seq_len, dim)
x = torch.randint(vocab_size, (bsz, seq_len))
logits, KVCache = model(x)
print(x.shape)
print(logits.shape)
print(KVCache[0].shape)

torch.Size([1, 20])
torch.Size([1, 20, 100])
torch.Size([1, 20, 16])


## Chunked-Prefill 实现

chunk 概念与page相似

In [8]:
x_chunks = x.split(page_size, dim = 1)
print(x_chunks)
KVCache = torch.zeros(2, bsz, seq_len, dim)

(tensor([[75, 23, 74,  9, 17, 30, 47, 94]]), tensor([[27, 46, 15, 15, 93, 36, 75, 78]]), tensor([[ 9, 69, 90, 45]]))


编写 chunk-prefill, 但是以下代码有什么逻辑问题？

In [9]:
def chunk_prefill_method(model, x, page_size, dim, ):

    bsz, seq_len = x.shape
    x_chunks = x.split(page_size, dim = 1)
    KVCache = torch.zeros(2, bsz, seq_len, dim)
    
    num_chunks = len(x_chunks)
    for i, x_c in enumerate(x_chunks):
        bsz, cur_len = x_c.shape
        logits, tmp_KVCache = model(x_c)
    
        if i == num_chunks-1: # 最后一个chunk
            last_token_logits = logits[:, -1, :]
            KVCache[0, :, i*page_size : i*page_size+cur_len, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size : i*page_size+cur_len, :] = tmp_KVCache[1]
        else:
            last_token_logits = None # 非最后一个 chunk 输出的 logits 是无效的
            KVCache[0, :, i*page_size : (i+1)*page_size, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size : (i+1)*page_size, :] = tmp_KVCache[1]
    return last_token_logits, KVCache

logits, KVCache = chunk_prefill_method(model, x, page_size, dim)
print(logits.shape)
print(KVCache.shape)

torch.Size([1, 100])
torch.Size([2, 1, 20, 16])


以上的代码问题在于，第 2 个 chunk 计算时，并没有将 第 1 个 chunk-kvcache 输入

```
-    k1, k2, k3, k4
- q1  x,  x, 
- q2  x,  x, 
- q3          x,  x, 
- q4          x,  x, 
```

实际上在 prefill 3,4 时, 将 k1,k2 加载进来


```
-    k1, k2, k3, k4
- q1  x,  x, 
- q2  x,  x, 
- q3  *,  *,  x,  x, 
- q4  *,  *,  x,  x, 
```

此时我们遇到了一种介于 Prefill 和 Decoding 之间的计算模式，区分如下

1. Prefill: 输入 [chunk], kv cache [None]
2. Decoding: 输入 [next_token], kv cache[context]
3. Chunk-prefill: 输入 [chunk], kv cache [chunks]

重写注意力, 实际上写法与 Decoding-Forward 模式相同

In [10]:
class ChunkedPrefillAttention(Attention):
    def forward(self, X, KVCache):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        
        if KVCache != None:
            K_cache, V_cache = KVCache[0], KVCache[1]
            K_ = torch.cat([K_cache, K], dim = 1)  # cat seq_len dimension 
            V_ = torch.cat([V_cache, V], dim = 1)  # cat seq_len dimension 
        else:
            K_, V_ = K, V

        print('QKV shape', f'{Q.shape}, {K_.shape}, {V_.shape}')
        print('-'*10)
        
        Z = attention_kernel(Q, K_, V_)
        O = self.Wo(Z)
        return O, [K,V]

In [11]:
def chunk_prefill_method(model, x, page_size, dim, ):

    bsz, seq_len = x.shape
    x_chunks = x.split(page_size, dim = 1)
    KVCache = torch.zeros(2, bsz, seq_len, dim)
    
    num_chunks = len(x_chunks)
    for i, x_c in enumerate(x_chunks):
        bsz, cur_len = x_c.shape

        # chunk_prefill
        if i==0:
            chunk_kv_cache = None
        else:
            chunk_kv_cache = KVCache[:, :, :i*page_size]
            print('tmp_kv_cache.shape:', chunk_kv_cache.shape)
            
        logits, tmp_KVCache = model.forward(x_c, KVCache=chunk_kv_cache)
    
        if i == num_chunks-1: # 最后一个chunk
            last_token_logits = logits[:, -1, :]
            KVCache[0, :, i*page_size : i*page_size+cur_len, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size : i*page_size+cur_len, :] = tmp_KVCache[1]
        else:
            last_token_logits = None # 非最后一个 chunk 输出的 logits 是无效的
            KVCache[0, :, i*page_size : (i+1)*page_size, :] = tmp_KVCache[0]
            KVCache[1, :, i*page_size : (i+1)*page_size, :] = tmp_KVCache[1]
            
    return last_token_logits, KVCache

In [12]:
model = XDGModel(dim, 
                 vocab_size, 
                 attention_layer=ChunkedPrefillAttention)
x = torch.randint(vocab_size, (bsz, seq_len))
logits, KVCache = chunk_prefill_method(model, x, page_size, dim)
print(logits.shape)
print(KVCache.shape)

QKV shape torch.Size([1, 8, 16]), torch.Size([1, 8, 16]), torch.Size([1, 8, 16])
----------
tmp_kv_cache.shape: torch.Size([2, 1, 8, 16])
QKV shape torch.Size([1, 8, 16]), torch.Size([1, 16, 16]), torch.Size([1, 16, 16])
----------
tmp_kv_cache.shape: torch.Size([2, 1, 16, 16])
QKV shape torch.Size([1, 4, 16]), torch.Size([1, 20, 16]), torch.Size([1, 20, 16])
----------
torch.Size([1, 100])
torch.Size([2, 1, 20, 16])


## Part-3 Mix-PD-Request ChunkPrefill

上述计算逻辑里只处理 Prefill Request, 我们将混合现有的 Decoding Request 进行 step 推理。

定义每个 Step 进入一个请求, 每个 Step 长度为 chunksize * randn(10), 目标新生成token 10个当作终止条件

In [18]:
import random 

random.randint(32,100)



76

In [51]:
from collections import deque

num_request = 6 # 设最大处理 request << page_size 
max_new_tokens = 128

prompts=[]
chunk_prompts = [] # list[deque]
completions = []
reqeust_kv_cache = []
prefill_set = set()
decoding_set = set()
completed_set = set()
cur_len = []

for i in range(num_request):
    prompt_len = random.randint(32,100)
    prompt = torch.randint(vocab_size, size=(1, prompt_len))
    prompts.append(prompt)
    completions.append([])
    reqeust_kv_cache.append(None)
    prefill_set.add(i)
    cur_len.append(0)

for prompt in prompts:
    chunk_prompt = prompt.split(page_size, dim = 1)
    q = deque()
    for chunk in chunk_prompt:
        q.append(chunk[0])
    chunk_prompts.append(q)

print(len(chunk_prompts[0]))
print(len(chunk_prompts[1]))
print(len(chunk_prompts[2]))
for data in chunk_prompts[0]:
    print(data)

11
13
9
tensor([41, 11, 27, 11, 58, 66, 10, 80])
tensor([94, 60, 72, 29,  2, 88, 99, 51])
tensor([50, 64, 64,  9, 49, 76, 53, 12])
tensor([93, 29, 40, 83, 71, 12, 72, 31])
tensor([18,  5, 52, 50, 69, 92, 19, 93])
tensor([79, 70,  4, 95, 48, 50, 31, 55])
tensor([ 9, 52,  2, 83, 45, 19, 71, 49])
tensor([89, 10, 51, 58, 73, 35,  6, 97])
tensor([76, 73, 92, 18, 31, 78, 33, 65])
tensor([53, 22, 66, 83, 57, 46, 32, 90])
tensor([30, 57, 68, 17, 73, 92])


In [46]:
from collections import deque

q = deque()
q.append(1)
q.append(3)
q.popleft()
q

deque([3])

In [44]:
a = set()
a.add(1)
a.add(2)
print(list(a))
len(a)
for k, i in enumerate(a):
    print(k, i)

[1, 2]
0 1
1 2


## 主循环

step1: 组batch技巧

1. decoding: 对 KVCache 需要 padding, input tensor 为 `1, max_decoding_batch`
2. prefill: 对 输入需要 padding. `n, page_size`

step2: 

In [23]:
# step

while 1:

    # 1. get batch
    # get decoding batch and kvcache
    # get prefill input and padding
    decoding_batch_to_request={}
    prefill_batch_to_request={}
    len_decoding = len(decoding_set)
    len_prefill = len(prefill_set)
    if len(decoding_set) == 0:
        batch_decoding_kvcache = None
        batch_input_ids_decoding = None
    else:
        batch_decoding_kvcache = []
        # batch_input_ids_decoding = torch.zeros( 1, len(decoding_set), dtype=torch.long) # 当成 bsz=1， seq_len=decoding_request_num
        batch_input_ids_decoding = torch.zeros( 1, page_size, dtype=torch.long) # 当成 bsz=1， seq_len=decoding_request_num
        for k, i in enumerate(decoding_set):
            next_token = chunk_prompts[-1][-1]
            
            batch_input_ids_decoding[0][k] = next_token
            
            tmp_kv_cache = reqeust_kv_cache[i]
            
            batch_decoding_kvcache.append(tmp_kv_cache) # 每个 request 的  KVCache 长度不同需要进一步 Right padding
            decoding_batch_to_request[k] = i 
            
    if len(prefill_set) == 0:
        batch_prefill_kvcache = None
        batch_input_ids_prefill = None
    else:
        batch_input_ids_prefill = torch.zeros(len_prefill, page_size,  dtype=torch.long)
        batch_prefill_kvcache = []
        last_pos = []
        for k, i in enumerate(prefill_set):

            if chunk_prompts[i] == 1:
                is_last_chunk = True
            else:
                is_last_chunk = False
                last_chunk_pos.append(-1)
            
            if len(chunk_prompts[i]) != 0:
                tmp_chunks = chunk_prompts[i].popleft()

                if is_last_chunk:
                    if tmp_chunks.shape[0] != page_size:
                        padding_len = page_size - tmp_chunks.shape[0]

                        last_pos.append(tmp_chunks.shape[0]-1)
                        
                        padding_input_ids = torch.zeros(padding_len)
                        tmp_chunks = torch.cat( (tmp_chunks, padding_input_ids), dim = 0)

                chunk_prefill_kvcache = reqeust_kv_cache[i]
                batch_prefill_kvcache.append(chunk_prefill_kvcache)
                
                batch_input_ids_prefill[k] = tmp_chunks # 最后一个块数据需要 padding 成 page_size
                prefill_batch_to_request[k] = i

    # merget batch
    batch_input_ids = torch.cat(
        # decoding: bsz = 1, seq_len = page_size
        # prefill:  bsz = prefill_request, seq_len = page_size
        (batch_input_ids_decoding, batch_input_ids_prefill), 
    )


    # 2. chunk-prefill forward step
    # get-logits and predict result
    logits, kv_decoding, kv_prefill=model(batch_input_ids, 
                                          batch_decoding_kvcache, 
                                          batch_prefill_kvcache)

    decoding_logits, prefill_logits = logits[0,:], logits[1:]
    next_token = torch.argmax( decoding_logits[0, :len_decoding, :], dim = -1)


    # 3. update 
    update_decoding_token = {}
    for i in len_decoding:
        request_id = decoding_batch_to_request(i)
        token = next_token[i]

        tmp_kvcache = kv_decoding[i].unsqueeze(dim=0)
        if reqeust_kv_cache[request_id] is None:
            reqeust_kv_cache[request_id] = tmp_kvcache
        else
            reqeust_kv_cache[request_id] = torch.cat( (reqeust_kv_cache[request_id], 
                                                       tmp_kvcache), dim = 0)
        
        
        update_decoding_token[request_id] = token
    for i in len_prefill:
        request_id = prefill_batch_to_request(i)
        if last_pos[i] == -1:
            continue
        else:
            token = torch.argmax( decoding_logits[i, :last_pos[i], :], dim = -1)
            update_decoding_token[request_id] = token

        tmp_kvcache = kv_prefill[i]
        if reqeust_kv_cache[request_id] is None:
            reqeust_kv_cache[request_id] = tmp_kvcache
        else
            reqeust_kv_cache[request_id] = torch.cat( (reqeust_kv_cache[request_id], 
                                                       tmp_kvcache), dim = 1) # seq level
            
    for i in range(num_request):
        # 更新请求状态
        if i in update_decoding_token:
            completions.append(update_decoding_token[i])
            
            if len(completions[i]) == 0:
                prefill_set.remove(i)
                decoding_set.add(i)

            if len(completions[i]) == 10: # 预设最大解码长度为 10, 终止生成
                decoding_set.remove(i)
                completed_set.add(i)
        
            

deque([1, 2, 3])


## Chunked Prefill 对推理系统设计

1. 找到各模块共性计算，进行融合 PD
2. 上述例子描述了 proj 类算子，我们需要进一步分析，注意力计算是否有类似的PD计算共性。

所幸 PD 的注意力遵循：

1. Decoding `单 q 多 KV`
2. Prefill 虽然注意力是 `多 q 多 KV` 计算的， 但其子任务仍为 `单 q 多 KV`

为了节省存储开销，原本所遵循的 kernel 区分了

```
forward_prefill(), page_attention_prefill_kernel()
forward_decoding(), page_attention_decoding_kernel()
```

目标要实现一种不区分 PD 的 通用kernel

```
forward_chunk_prefill(), page_attention_kenrl()
```

恭喜你，发明了 vLLM-V1 版本的 engine step， 计算迭代时间部是不分 PD 任务的。

vLLM-V1 的另外一个特性，乃至是 Inference 系统，会设计成 PD 分离架构，

1. Prefill节点：采用 Chunk-Prefilled 或 standard-Prefilled
2. Decoding节点：对应硬件采用更高速的通信接口，提高访存效率。相应的 Decoding 节点对计算的要求是低于 prefill 节点的。常对 Decoding 节点上更大的 batch size，提高 batch-decoding 效率。

## vLLM-V1 实现逻辑

1. PD 注意力共性计算可以单独优化 Kernel 和 step 流程
2. Chunk-Prefill 仅是一种高效的投影矩阵乘算法

我们将按照以下顺序更新 vLLM-V1

1. PD 混合的 PageAttention kernel，
2. 结合 Chunk-Prefill 实现完整工程
3. 实现 PD 分离